# List of Functions

In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from numpy import loadtxt
from scipy.fftpack import fft
from scipy.fft import fft, ifft, fftfreq
from numpy import arange
from skimage.measure import EllipseModel
from matplotlib.patches import Ellipse
from scipy.signal import savgol_filter
from random import randint
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from sklearn import svm
import sys

## Miselaneous Functions

In [ ]:
def set_plot_style(style_num = 17):
    # This function sets the style of the images that are generated by matplotlib. 
    plt.style.use('default')
    plt.style.use(plt.style.available[style_num])

    if style_num == 'default': 
        plt.style.use('default')

def suffle_data(dataset_data, dataset_labels):
    # This function shuffles both the dataset_data and dataset_labels based on the same 
    # random permutation order. This function is useful for shuffling data and labels while maintaining 
    # the association between a data array and its label.
    num_examples = dataset_data.shape[0]
    shuffle_idx = np.random.permutation(num_examples)
    suffled_dataset = dataset_data[shuffle_idx, :]
    suffled_labels = dataset_labels[shuffle_idx]
    
    return suffled_dataset, suffled_labels

def suffle_index(dataset_arrays):
    # This functions suffles the measurements in an array full of bacteria measurments.
    # That is, it suffles the index position of each measurement inside a bigger array. This is useful 
    # when is needed to shuffle a single array without any corresponding labels.
    num_examples = dataset_arrays.shape[0]
    shuffle_idx = np.random.permutation(num_examples)
    suffled_dataset_arrays = dataset_arrays[shuffle_idx, :]
    
    return suffled_dataset_arrays

## Data Preprocesing for imported signals

### Fix Signal Shifting Function
Here a manipulation of the data is performed, in order to fix the signal shifting that exists in the data. Thus, a linear regresion is accomplished to flatten down the signal and to center the signal at zero. In addition, the displacement of the graphene layer $z(t) = y_{nm}$ is calculated using the following equation:

$y(t) = [{V_{pd}(t)/〈V_{pd}(t)〉− 1}]/\phi$

In [ ]:
def fix_signal_shifting(signals_arrays, tf):
    # This function takes an array full with multiple measurements. For each measurement case,
    # the signal is flattened and centered at zero. It returns the fixed signals with their time vector. 
    # Inputs: signals_arrays is the array containing all the signal arrays and tf is the time length of each
    # measured signal.

    time_array = np.linspace(0, tf, signals_arrays.shape[1])
    fixed_signal_arrays = np.empty(shape=[signals_arrays.shape[0], signals_arrays.shape[1]])

    for signal_array in range(signals_arrays.shape[0]):
        P = np.polyfit(time_array,signals_arrays[signal_array,:],1)
        yplot = signals_arrays[signal_array,:] - P[0]*time_array - P[1]
        fixed_signal = (yplot/np.mean(signals_arrays[signal_array,:]))/ 0.0038;  
        fixed_signal_arrays[signal_array] = fixed_signal 
    return fixed_signal_arrays, time_array

### Function to filter signals by variance

In [ ]:
def filter_by_variance_threshold(signals_arrays, tf, threshold_range):
    # This function takes an array of arrays of signals and calculates the variance of each signal.
    # Then, it saves in a new array all cases that have a variace below a desired threshold.
    # Inputs: signals_arrays is the arrays of signal arrays and threshold is the desition 
    # variance threshold, which can be defined as a range in a python list.
    time_array = np.linspace(0, tf, signals_arrays.shape[1])
    variances = np.var(signals_arrays, axis=1)
    lower_bound, upper_bound = threshold_range
    filtered_indices = np.where((variances >= lower_bound) & (variances < upper_bound))[0]
    filtered_data = signals_arrays[filtered_indices, :]
    return filtered_data, time_array

### For both flattening - fixing the drift in data and for filtering the data by variance

In [ ]:
def flatten_fix_n_filter_data(signals_arrays, tf, flatten, filter_by_variance, threshold_range = None):
    # This function takes an array full with multiple measurements to flatten and fix the shifting in the signals by using the 
    # fix_signal_shifting() function. Additionally, one can filter the data by variance ranges. 
    if flatten and filter_by_variance:
        fixed_n_flatten, time = fix_signal_shifting(signals_arrays, tf)
        final_signals_arrays, time = filter_by_variance_threshold(fixed_n_flatten, tf, threshold_range)
    elif flatten:
        final_signals_arrays, time = fix_signal_shifting(signals_arrays, tf)
    elif filter_by_variance:
        final_signals_arrays, time = filter_by_variance_threshold(signals_arrays, tf, threshold_range)
    else:
        final_signals_arrays = signals_arrays
        time = np.linspace(0, tf, signals_arrays.shape[1])
    
    if threshold_range != None and filter_by_variance == False:
        print('You have written a threshold_range but filter_by_variance = False. Please change it if you want to filter in this range.')

    print(f'The data used has a shape of {final_signals_arrays.shape}')

    return final_signals_arrays, time

## Plot time-domain signals

In [ ]:
def plot_time_signal(signal_array, tf, title, file_name, xlim, ylim, save = False, style_num = 17):
    # This function takes the array information to plot the time series information of a single signal.
    t = np.linspace(0, tf, len(signal_array))

    set_plot_style(style_num)
    plt.plot(t, signal_array, color="xkcd:salmon", linewidth=0.5)
    plt.xlabel('Time [s]')
    plt.ylabel('Motion [nm]')

    if title != 'none':
        plt.title('Time signal graph of ' + title)
        plt.title(title)
    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if ylim != 'auto':
        plt.ylim(ylim[0], ylim[-1])
    if save:
        plt.savefig(f'time_plot_for_{file_name}.jpg', dpi=300)
    plt.show()

### Filter signal with Savgol filter

In [16]:
from scipy.signal import savgol_filter

def filter_signals(signal_array, window_length_val, polyorder_val):
    # This function takes an array of signals and filters them.
    # Then it creates a new array with all the filtered signals in the same order.
    
    signals_filtered = np.zeros((signal_array.shape[0], signal_array.shape[1]))
    for idx, signal in enumerate(signal_array):
        ynm = savgol_filter(signal, window_length= window_length_val, polyorder =polyorder_val)
        signals_filtered[idx] = ynm
    
    return signals_filtered

## Plot Variance boxplots

In [ ]:
def plot_horizontal_boxplot_variances(*datasets, labels=None, title='none', file_name='variances', xlim='auto', save=False, style='default'):
    # Plots horizontal boxplots of the variances of multiple datasets (variance computed row-wise),
    # applying a predefined plot style and custom colors. Returns summary statistics (min, Q1, median,
    # Q3, max) for each dataset. Allows setting labels, title, x-axis limits, and saving the figure.
    
    num_datasets = len(datasets)
    fig, ax = plt.subplots()

    set_plot_style(style_num = 17)

    variances_list = []
    summary_stats_list = []

    for i, dataset in enumerate(datasets):
        variances = np.var(dataset, axis=1)
        variances_list.append(variances)

        # Calculate summary statistics
        Q0 = np.min(variances)
        Q1 = np.percentile(variances, 25)
        Q2 = np.percentile(variances, 50)
        Q3 = np.percentile(variances, 75)
        Q4 = np.max(variances)
        summary_stats_list.append({'Q0': Q0, 'Q1': Q1, 'Q2': Q2, 'Q3': Q3, 'Q4': Q4})

    # Plot box plots
    boxplot_dict = ax.boxplot(variances_list, vert=False, patch_artist=True)

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    for patch, color in zip(boxplot_dict['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
        patch.set_linewidth(1)

    for whisker in boxplot_dict['whiskers']:
        whisker.set(color='black', linewidth=1)

    for cap in boxplot_dict['caps']:
        cap.set(color='black', linewidth=1)

    for median in boxplot_dict['medians']:
        median.set(color='black', linewidth=2)

    for flier in boxplot_dict['fliers']:
        flier.set(marker='o', markersize=5, markeredgecolor='black', markerfacecolor='black', alpha=0.5)

    ax.set_xlabel('Variance (${nm}^{2}$)', fontsize=12)
    ax.tick_params(axis='both', labelsize=10)

    if title != 'none':
        ax.set_title(title, fontsize=14)
    ax.set_yticklabels(labels or [f'Case {i+1}' for i in range(num_datasets)])
    
    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if save:
        plt.savefig(f'specgram_{file_name}.jpg', dpi=300)
    plt.show()

    return summary_stats_list


## Calculation of normal PSD vs frequency plot

In [ ]:
from scipy.fftpack import fft
from scipy.signal import welch

def get_PSD_graph(signal, tf, title, file_name, dpi_val, PSD_style = 'normal_PSD', xlim = 'auto', ylim = 'auto', save = False):
    
    set_plot_style(style_num=17)
    tf = ecoli7744_CTR_df['Time [s]'][0]
    ts = np.linspace(0, tf, len(signal))
    fs = len(signal) / ts[-1]
    T = 1.0 / fs

    # Number of sample points
    N = len(signal)

    # FFT
    yf = fft(signal)

    if PSD_style == 'normal_PSD':
        # Compute the frequencies
        xf = np.linspace(0.0, 1.0/(2.0*T), N//2)
        # Compute the power spectral density
        psd_fft = 2.0/N * np.abs(yf[0:N//2])**2
        # Plot FFT-based power spectral density
        plt.grid()
        plt.loglog(xf, psd_fft, color = 'red', alpha = 0.3)
        if title != 'none':
            plt.title('Power Spectral Density (FFT) ' + title)
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Power Spectral Density [${nm}^{2}$ ${Hz}^{-1}$]') #[V**2/Hz]
        plt.ylim(1e-6, 1e7)
        plt.xlim(1e-1, 1e3)
    elif PSD_style == 'welch_PSD':
        frequencies, psd_welch = welch(signal, fs)
        # plt.figure(figsize=(10, 5))
        plt.semilogx(frequencies, psd_welch, color = 'red', alpha = 0.8)
        if title != 'none':
            plt.title('Power Spectral Density (Welch) ' + title)
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Power Spectral Density [${nm}^{2}$ ${Hz}^{-1}$]') #[V**2/Hz]
        plt.grid()

    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if ylim != 'auto':
        plt.ylim(ylim[0], ylim[-1])
    if save:
        plt.savefig('PSD_' + file_name + '.jpg', dpi=dpi_val)
    plt.show()

## Calculation of the Short-time Fourier transform (STFT)

Functions to perform the calculation of the STFT in order to create a spectogram image that could be used in a machine learning algorithm to classify bacteria. 

In [ ]:
from scipy import signal
from skimage.measure import block_reduce
from scipy.signal import spectrogram

def plot_signal_stft(phase_data, tf, ylim, title, file_name, plot = False, save = False):
    # This function receives one single signal and computes the stft and plots the spectrogram
    # with the signal.stft() funtion. You can choose between plotting or saving the picture as well. It also 
    # returns the shape of the picture. 
    # Inputs: phase_signal is the single array of the bacteria signal and tf is the length in time of the signal.
    
    ts = np.linspace(0,tf,len(phase_data))
    fs = len(phase_data)/ts[-1] #Number of samples divided by the mesured time for the samplying frequency.
    
    f_func, t_func, Zxx = signal.stft(phase_data, fs, boundary = None)
    
    reducedZ = block_reduce(np.abs(Zxx), block_size=(1, 1), func=np.max)
    reducedf = f_func[0::1]
    reducedt = t_func[0::1]
    
    reducedZ = np.log(reducedZ)
    
    if plot:
        set_plot_style() #Sets the style of the images back to Style 9 used in other parts of code
        plt.pcolormesh(reducedt, reducedf, reducedZ, shading='auto') 
        plt.title('STFT ' + title)
        plt.ylabel('Frequency [Hz]')
        plt.xlabel('Time [s]')
        if ylim != 'none':
            plt.ylim(ylim)
        if save:
            plt.savefig('stft_' + file_name +'.jpg', dpi=300)
        plt.show()
        set_plot_style(style_num = 9) #Sets the style of the images back to Style 9 used in other parts of code
    return reducedZ.shape, Zxx

def get_stft_all_img_arrays(signals_array, tf): 
    # This function gets an array of bacteria signals and calculates the stft, it creates an image out of it and 
    # then converts the images into RGB arrays. Finally, it puts every array in a bigger stacked array. 
    # Inputs: signals_array: Is the complete array of a bacteria type / tf is the length in time of each signal in the signal arrays

    for i in range(signals_array.shape[0]):
        
        ts = np.linspace(0,tf,len(signals_array[i]))
        fs = len(signals_array[i])/ts[-1]
          
        f_func, t, Zxx = signal.stft(signals_array[i], fs, boundary = None)
    
        reducedZ = block_reduce(np.abs(Zxx), block_size=(1, 1), func=np.max) #this is the shape of the image
        reducedf = f_func[0::1]
        reducedt = t[0::1]
        
        reducedZ = np.log(reducedZ) #This is taking the logarithm of the data
        
        set_plot_style() # Set the plot style. This is by default style 17. 
        fig = plt.pcolormesh(reducedt, reducedf, reducedZ, shading='auto')
        plt.axis('off')

        image_array = fig.to_rgba(fig.get_array().reshape(reducedZ.shape))
        current_fig = plt.gcf()
        plt.close(current_fig)
                   
        if i == 0:
            images_arrays = np.zeros((signals_array.shape[0],reducedZ.shape[0],reducedZ.shape[1],4))
            
        images_arrays[i,:,:,:] = image_array
        
        set_plot_style(style_num = 9) # #Sets the style of the images back to Style 9 used in other parts of code
    return images_arrays

### Plot Spectogram images

In [ ]:
import random

def generate_random_numbers(n=10, max_value=100):
    # This fucntion generates a series of random numbers. 
    random_numbers = []
    for _ in range(n):
        random_number = random.randint(0, max_value)
        random_numbers.append(random_number)
    return random_numbers

def get_some_img_in_arrays(train_or_test_set, labels_array, class_names, file_name = None, num_img = 10, vmin_val = None, vmax_val = None, colorbar_active = False, grayscale_active = False, save = False):
    # This function plots a set of spectrograms to visualize how this info is being inputted in the ML algorithms.
    indexes = generate_random_numbers(n=num_img, max_value=train_or_test_set.shape[0]-1)

    # Show 10 random images:
    plt.figure(figsize=(20,20))
    for i in range(num_img):
        plt.subplot(5,5,i+1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)

        if vmin_val != None and vmax_val != None:
            plt.imshow((train_or_test_set[indexes[i]]*255).astype(np.uint8), aspect='auto', origin='lower', vmin = vmin_val, vmax = vmax_val) 
        else:
            plt.imshow((train_or_test_set[indexes[i]]*255).astype(np.uint8), aspect='auto', origin='lower')

        if grayscale_active:
            plt.imshow((train_or_test_set[indexes[i]]*255), aspect='auto', origin='lower', cmap='gray', vmin=0, vmax=255)

        if colorbar_active:
            plt.colorbar()
        plt.xlabel(class_names[int(labels_array[indexes[i]])])
    if save:
        plt.savefig('specgram_' + file_name + '.jpg', dpi=300)
    plt.show()

## Functions for STFT calculated with plt.specgram() from Matplotlib

In [ ]:
def get_specgram_all_img_arrays(signals_array, tf): 
    # This function gets an array of bacteria signals and calculates the stft by using the plt.specgram() function 
    # it creates an image out of it and then converts the images into RGB arrays. Finally, it puts every array in a bigger stacked array. 
    # Inputs: signals_array: Is the complete array of a bacteria type and tf is the length in time of each signal in the signal arrays.
    
    for i in range(signals_array.shape[0]):
        
        ts = np.linspace(0,tf,len(signals_array[i]))
        fs = len(signals_array[i])/ts[-1]

        set_plot_style() # Set the plot style. This is by default style 17. 
        Pxx, freqs, bins, im = plt.specgram(signals_array[i],Fs=fs)
        plt.grid(False)
        plt.axis('off')

        image_array = im.to_rgba(im.get_array()) #creates a matrix of numbers from images.
        current_fig = plt.gcf()
        plt.close(current_fig)

        if i == 0:
            images_arrays = np.zeros((signals_array.shape[0],image_array.shape[0],image_array.shape[1],image_array.shape[2]))

        images_arrays[i,:,:,:] = image_array
        
        set_plot_style(style_num = 9) #Sets the style of the images back to Style 9 used in other parts of code
    return images_arrays

In [ ]:
def plot_single_spectrogram(phase_data, tf, title, file_name, nperseg_val, noverlap_val, cmap_type, dpi_val, ylim='auto', xlim='auto', save=False, axis=True, colorbar=True, normalize=True, log_scale=False, filter_freq = False, Fxx_grayscale = False, vmin=None, vmax=None):
    # This function takes a single array of a bacteria signal and calculates the STFT to later create an image of the spectrogram by using
    # the scipy.signal.spectrogram() function instead of the plt.specgram() used in "plot_single_specgram()".
    # Inputs: phase_data is the array of a single signal of a bacteria, tf is the duration in time of the signal, nperseg_val is the number of elements
    # used per window, noverlap_val is the number of overlaped elements if windeos are desired to be overlaped, cmap_type is the type of cmap color used.
    ts = np.linspace(0, tf, len(phase_data))
    fs = len(phase_data) / ts[-1]

    # apply notch filter to phase data
    if filter_freq:
        phase_data = notch_filter(phase_data, 100, fs) 

    set_plot_style() 

    freqs, bins, Pxx = spectrogram(phase_data, fs=fs, nperseg=nperseg_val, noverlap=noverlap_val)

    if log_scale:
        Pxx = 10 * np.log10(Pxx)  # Convert to logarithmic scale

    if normalize:
        Pxx_normalized = (Pxx - np.min(Pxx)) / (np.max(Pxx) - np.min(Pxx))
        data_to_plot = Pxx_normalized
    else:
        data_to_plot = Pxx

    im = plt.imshow(data_to_plot, aspect='auto', origin='lower', extent=[ts[0], ts[-1], freqs[0], freqs[-1]], cmap=cmap_type, vmin=vmin, vmax=vmax)
    
    if Fxx_grayscale:
        im = plt.imshow((data_to_plot*255), aspect='auto', origin='lower', extent=[ts[0], ts[-1], freqs[0], freqs[-1]], cmap='gray', vmin=0, vmax=255)

    plt.ylabel('Frequency [Hz]')
    plt.xlabel('Time [s]')
    plt.xlim(ts[0], ts[-1])

    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if title != 'none':
        plt.title('STFT ' + title)
    if ylim != 'auto':
        plt.ylim(ylim[0], ylim[-1])
    if axis == False:
        plt.axis('off')
    if colorbar == True:
        cbar = plt.colorbar(im, format='%.1f')
        if log_scale:
            cbar.set_label('PSD [dB]')
        else:
            cbar.set_label(r'PSD [$nm^{2} Hz^{-1}$]')
        ticks = np.linspace(im.norm.vmin, im.norm.vmax, num=8)  # generate equally spaced ticks between min and max
        cbar.set_ticks(ticks)
    if save:
        plt.savefig('specgram_' + file_name + '.jpg', dpi=dpi_val)
    plt.show()
    set_plot_style(style_num = 9) 
    window_duration = round(nperseg_val/fs, 3) 
    overlap_duration = round(noverlap_val/fs, 3)


    return im, Pxx, ('Window size (s): ' + str(window_duration), 'Overlap size (s): ' + str(overlap_duration))

def plot_spectrogram_groups(signals_arrays, tf, titles_list, file_name_list, nperseg_val, noverlap_val, cmap_type, dpi_val, xlim, ylim, log_scale = False, normalize = False, colorbar = True, filter_freq = False, same_color_scale = True, find_scale = None, save = False):
    # This function takes an array of a bacteria signal cases and calculates the STFT to later create an image of the spectrogram by using
    # the scipy.signal.spectrogram() function for each of the inputed cases.
    # Inputs: signals_arrays is the array containing the measurements of a bacteria, tf is the duration in time of the signals, nperseg_val is the number of elements
    # used per window, noverlap_val is the number of overlaped elements if windeos are desired to be overlaped, cmap_type is the type of cmap color used.
    Pxx_list = []
    global_min, global_max = np.inf, -np.inf
    min_PSD_val, max_PSD_val = [], []
    vmin_val = None
    vmax_val = None 

    # Calculate Pxx values for all signals and find global minimum and maximum
    for signal_array in signals_arrays:
        ts = np.linspace(0, tf, len(signal_array))
        fs = len(signal_array) / ts[-1]
        freqs, bins, Pxx = spectrogram(signal_array, fs=fs, nperseg=nperseg_val, noverlap=noverlap_val)
        Pxx_list.append(Pxx)

        if same_color_scale:
            if find_scale == 'global_minmax':
                global_min, global_max = min(global_min, np.min(Pxx)), max(global_max, np.max(Pxx))
            if find_scale == 'average_minmax':
                min_PSD_val.append(np.min(Pxx))
                max_PSD_val.append(np.max(Pxx))
    
    if same_color_scale:
        if find_scale == 'global_minmax':
            vmin_val = global_min
            vmax_val = global_max
        elif find_scale == 'average_minmax':
            vmin_val = np.mean(min_PSD_val)
            vmax_val = np.mean(max_PSD_val)
    
    images_arrays = np.zeros((signals_arrays.shape[0], 234, 234, 4))

    if log_scale:
        vmin_val = 10*np.log10(vmin_val)
        vmax_val = 10*np.log10(vmax_val)

    # Generate spectrogram images with the same color scale
    for i, signal in enumerate(signals_arrays):
        ts = np.linspace(0, tf, len(signal))
        fs = len(signal) / ts[-1]

        set_plot_style() #automatically set to matplotlib style number 17

        Pxx_display = Pxx_list[i]
        if log_scale:
            Pxx_display = 10 * np.log10(Pxx_display)  #Convert to logarithmic scale
        if normalize:
            Pxx_display = (Pxx_display - np.min(Pxx_display)) / (np.max(Pxx_display) - np.min(Pxx_display))

        plt.figure()  # Create a new figure for each signal
        im = plt.imshow(Pxx_display, aspect='auto', origin='lower', extent=[ts[0], ts[-1], freqs[0], freqs[-1]], cmap=cmap_type, vmin=vmin_val, vmax=vmax_val)
        plt.title('STFT for ' + titles_list[i])
        plt.ylabel('Frequency [Hz]')
        plt.xlabel('Time [s]')
        plt.xlim(ts[0], ts[-1])

        if xlim != 'auto':
            plt.xlim(xlim[0], xlim[-1])
        if ylim != 'auto':
            plt.ylim(ylim[0], ylim[-1])
        if colorbar == True:
            cbar = plt.colorbar(im, format='%.1f')
            if log_scale:
                cbar.set_label('PSD [dB]')
            else:
                cbar.set_label(r'PSD [$nm^{2} Hz^{-1}$]')
            ticks = np.linspace(im.norm.vmin, im.norm.vmax, num=8)  #generate equally spaced ticks between min and max
            cbar.set_ticks(ticks)
        if save:
            plt.savefig('specgram_' + file_name_list[i] + '.jpg', dpi=dpi_val)
        plt.show()

In [ ]:
import cv2
def get_all_stft_spectrogram_rgb_arrays(signals_array, tf, nperseg_val, noverlap_val, log_scale, normalize, max_freq='auto', vmin_val = None, vmax_val = None, cmap_type='viridis', same_color_scale=True, find_scale = None):
    Pxx_list = []
    global_min, global_max = np.inf, -np.inf
    min_PSD_val, max_PSD_val = [], []
    vmin_val = None 
    vmax_val = None 

    # Calculate Pxx values for all signals and find global minimum and maximum
    for signal in signals_array:
        ts = np.linspace(0, tf, len(signal))
        fs = len(signal) / ts[-1]
        freqs, _, Pxx = spectrogram(signal, fs=fs, nperseg=nperseg_val, noverlap=noverlap_val)
        Pxx_list.append(Pxx)

        if same_color_scale:
            if find_scale == 'global_minmax':
                global_min, global_max = min(global_min, np.min(Pxx)), max(global_max, np.max(Pxx))
            if find_scale == 'average_minmax':
                min_PSD_val.append(np.min(Pxx))
                max_PSD_val.append(np.max(Pxx))

    #And now for all spectrograms: 
    if same_color_scale:
        if find_scale == 'global_minmax':
            vmin_val = global_min
            vmax_val = global_max
        elif find_scale == 'average_minmax':
            vmin_val = np.mean(min_PSD_val)
            vmax_val = np.mean(max_PSD_val)

    images_arrays = np.zeros((signals_array.shape[0], 234, 234, 4))

    if log_scale:
        vmin_val = 10*np.log10(vmin_val)
        vmax_val = 10*np.log10(vmax_val)

    # Generate spectrogram images with the same color scale
    for i, signal in enumerate(signals_array):
        ts = np.linspace(0, tf, len(signal))
        fs = len(signal) / ts[-1]

        set_plot_style() #automatically set to matplotlib style number 17

        Pxx_display = Pxx_list[i]
        if log_scale:
            Pxx_display = 10 * np.log10(Pxx_display)  # Convert to logarithmic scale
        if normalize:
            Pxx_display = (Pxx_display - np.min(Pxx_display)) / (np.max(Pxx_display) - np.min(Pxx_display))

        if max_freq != 'auto':
            max_index = np.searchsorted(freqs, max_freq)
            Pxx_display = Pxx_display[:max_index, :]
            freqs = freqs[:max_index]

        im = plt.imshow(Pxx_display, aspect='auto', origin='lower', extent=[ts[0], ts[-1], freqs[0], freqs[-1]], cmap=cmap_type, vmin=vmin_val, vmax=vmax_val)

        plt.grid(False)
        plt.axis('off')

        image_array = im.to_rgba(im.get_array())
        plt.close()

        image_array_resized = cv2.resize(image_array, (234, 234), interpolation=cv2.INTER_LINEAR)

        images_arrays[i] = image_array_resized

        set_plot_style(style_num=9) 

    return images_arrays, vmin_val, vmax_val

## This function to use the Pxx matrix

The Pxx matrix is the one that contains the PSD information of a signal. 

In [25]:
def get_all_Pxx_matrices(signals_array, tf, nperseg_val, noverlap_val, log_scale, vmin_val = None, vmax_val = None, same_color_scale=True, limit_Pxx_height = None, find_scale = None):

    global_min, global_max = np.inf, -np.inf
    min_PSD_val, max_PSD_val = [], []
    
    # Temporary list to store the modified matrices
    temp_Pxx_list = []

    for i, signal in enumerate(signals_array):
        ts = np.linspace(0, tf, len(signal))
        fs = len(signal) / ts[-1]
        freqs, _, Pxx = spectrogram(signal, fs=fs, nperseg=nperseg_val, noverlap=noverlap_val)
        
        if log_scale:
            Pxx = np.log1p(Pxx)

        if limit_Pxx_height is not None:
            Pxx = Pxx[:limit_Pxx_height, :]

        temp_Pxx_list.append(Pxx)
        
        if same_color_scale:
            if find_scale == 'global_minmax':
                global_min, global_max = min(global_min, np.min(Pxx)), max(global_max, np.max(Pxx))
            elif find_scale == 'average_minmax':
                min_PSD_val.append(np.min(Pxx))
                max_PSD_val.append(np.max(Pxx))

    # Convert the list back to a numpy array
    Pxx_array = np.array(temp_Pxx_list)

    if same_color_scale:
        if find_scale == 'global_minmax':
            vmin_val = global_min
            vmax_val = global_max
        elif find_scale == 'average_minmax':
            vmin_val = np.mean(min_PSD_val)
            vmax_val = np.mean(max_PSD_val)

        # Applying the scaling to all Pxx matrices
        for i in range(len(temp_Pxx_list)):
            temp_Pxx_list[i] = (temp_Pxx_list[i] - vmin_val) / (vmax_val - vmin_val)

        # Convert the list back to a numpy array again after scaling
        Pxx_array = np.array(temp_Pxx_list)

    # if log_scale:
    #     vmin_val = 10*np.log10(vmin_val)
    #     vmax_val = 10*np.log10(vmax_val)

    return Pxx_array, vmin_val, vmax_val, freqs

## Accuracy and Loss calculation

In [ ]:
def plot_accuracy_n_loss(model_history, epoch_num, title, xlim, ylim, dpi_val, file_name, accuracy_plot = False, loss_plot = False, save = False):
    
    if accuracy_plot:
        set_plot_style(style_num = 17)
        plt.plot(model_history.history['accuracy'], label='Train Set Accuracy')
        plt.plot(model_history.history['val_accuracy'], label = 'Validation Set Accuracy') 
        plt.ylabel('Accuracy')
    if loss_plot:
        set_plot_style(style_num = 17)
        plt.plot(model_history.history['loss'], label='Train Set Loss') 
        plt.plot(model_history.history['val_loss'], label = 'Validation Set Loss')  
        plt.ylabel('Loss')

    plt.xlabel('Epochs')
    plt.xlim(0, epoch_num)
    plt.legend(loc='best')      

    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if title != 'none':
        plt.title(title)
    if ylim != 'auto':
        plt.ylim(ylim[0], ylim[-1])
    if save:
        plt.savefig('plot_of_' + file_name + '.jpg', dpi=dpi_val)
    plt.show()
    set_plot_style(style_num = 9) 


### To plot the average Accuracy and Average Loss in a K-Fold analysis

In [ ]:
def average_lists(list_of_lists):
    np_array = np.array(list_of_lists) #to convert the list of lists to a NumPy array
    average_array = np.mean(np_array, axis=0) #to calculate the average along the first axis
    average_list = average_array.tolist() #to convert the result back to a list
    return average_list

import numpy as np
import matplotlib.pyplot as plt

def plot_average_accuracy_n_loss(model_history, epoch_num, title, xlim, ylim, dpi_val, file_name, accuracy_plot = False, loss_plot = False, save = False):
    
    accuracy_list = []
    accuracy_val_list = []
    loss_list = []
    loss_val_list = []

    for training_instance in range(len(model_history)):
        accuracy_list.append(model_history[training_instance].history['accuracy'])
        accuracy_val_list.append(model_history[training_instance].history['val_accuracy'])
        loss_list.append(model_history[training_instance].history['loss'])
        loss_val_list.append(model_history[training_instance].history['val_loss'])

    # Calculate averages
    accuracy_epochs_average = average_lists(accuracy_list)
    accuracy_val_epochs_average = average_lists(accuracy_val_list)
    loss_epochs_average = average_lists(loss_list)
    loss_val_epochs_average = average_lists(loss_val_list)

    # Calculate standard deviations
    accuracy_std = np.std(accuracy_list, axis=0)
    accuracy_val_std = np.std(accuracy_val_list, axis=0)
    loss_std = np.std(loss_list, axis=0)
    loss_val_std = np.std(loss_val_list, axis=0)

    if accuracy_plot:
        set_plot_style(style_num = 17)
        plt.plot(accuracy_epochs_average, label='Train Set Accuracy')
        plt.fill_between(range(epoch_num), np.array(accuracy_epochs_average) - accuracy_std, np.array(accuracy_epochs_average) + accuracy_std, alpha=0.25)
        plt.plot(accuracy_val_epochs_average, label = 'Validation Set Accuracy')
        plt.fill_between(range(epoch_num), np.array(accuracy_val_epochs_average) - accuracy_val_std, np.array(accuracy_val_epochs_average) + accuracy_val_std, alpha=0.25)
        plt.ylabel('Accuracy')

    if loss_plot:
        set_plot_style(style_num = 17)
        plt.plot(loss_epochs_average, label='Train Set Loss')
        plt.fill_between(range(epoch_num), np.array(loss_epochs_average) - loss_std, np.array(loss_epochs_average) + loss_std, alpha=0.25)
        plt.plot(loss_val_epochs_average, label = 'Validation Set Loss')
        plt.fill_between(range(epoch_num), np.array(loss_val_epochs_average) - loss_val_std, np.array(loss_val_epochs_average) + loss_val_std, alpha=0.25)
        plt.ylabel('Loss')

    plt.xlabel('Epochs')
    plt.xlim(0, epoch_num)
    plt.legend(loc='best')      

    if xlim != 'auto':
        plt.xlim(xlim[0], xlim[-1])
    if title != 'none':
        plt.title(title)
    if ylim != 'auto':
        plt.ylim(ylim[0], ylim[-1])
    if save:
        plt.savefig('plot_of_' + file_name + '.jpg', dpi=dpi_val)
    plt.show()
    set_plot_style(style_num = 9) 


### Confusion Matrix

In [ ]:
import seaborn as sn
from sklearn.metrics import confusion_matrix

def plot_conf_matrix(model, test_info, test_labels, algorithm_type, title, file_name, target_names, save=False, normalize=False, check_probabilities=False):
    # This function uses the test data array to calculate the predictions of a trained model and plot the confusion matrix
    # It is possible to be used with different Machine Learning algorithms like SVM, normal Neural Networks and Convolutional Neural Networks.
    # Inputs: model is the trained model, test_info is the array of the test data, test labels are the names of the test data, algorithm_type
    # is the type of machine learnign algorithm used (NN, SVM).

    if algorithm_type == 'NN':
        y_predicted = model.predict(test_info)
        y_predicted_labels = [np.argmax(i) for i in y_predicted]
        cm = confusion_matrix(test_labels, y_predicted_labels)
        eval_model = model.evaluate(test_info, test_labels, verbose=0)
        accuracy_in_test_set = eval_model[1]

    elif algorithm_type == 'NN_one_output_neuron':
        y_predicted = model.predict(test_info)
        y_predicted_labels = np.round(y_predicted).flatten()
        cm = confusion_matrix(test_labels, y_predicted_labels)
        test_loss, accuracy_in_test_set = model.evaluate(test_info, test_labels, verbose=0)
        # Debugging: Print unique values in test_labels and y_predicted_labels
        print("Unique values in test_labels:", np.unique(test_labels))
        print("Unique values in y_predicted_labels:", np.unique(y_predicted_labels))

    elif algorithm_type == 'SVM':
        y_predicted_labels = model.predict(test_info)
        cm = confusion_matrix(test_labels, y_predicted_labels)
        accuracy_in_test_set = model.score(test_info, test_labels)

    elif algorithm_type == 'NN_encoded_labels':
        y_predicted = model.predict(test_info)
        y_predicted_labels = [np.argmax(i) for i in y_predicted]
        test_labels_binary = [np.argmax(i) for i in test_labels]  # Convert test_labels from one-hot encoded to binary format
        cm = confusion_matrix(test_labels_binary, y_predicted_labels)
        eval_model = model.evaluate(test_info, test_labels, verbose=0)
        accuracy_in_test_set = eval_model[1]
    
    if normalize:
        cmn = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        # Debugging: Print the raw confusion matrix
        print("Raw confusion matrix:\n", cm)
        cm_img = plt.figure(figsize=(4, 4))
        sn.heatmap(cmn, annot=True, fmt='.2f', xticklabels=target_names,
                   yticklabels=target_names, cmap='Blues', cbar=False)
    else:
        cm_img = plt.figure(figsize=(4, 4))
        sn.heatmap(cm, annot=True, fmt='d', xticklabels=target_names,
                   yticklabels=target_names, cmap='Blues', cbar=False)

    if check_probabilities == True and algorithm_type != 'SVM': #Check if the predicted probabilities sum up to 1
        normal = False
        tolerance = 1e-5
        sums = np.sum(y_predicted, axis=1)
        for i in range(len(sums)):
            if abs(sums[i] - 1) > tolerance:
                print("Warning: sum of predicted probabilities for input", i, "is not close to 1:", sums[i])
            else:
                normal = True
        if normal == True:
            print('No issues with the sum of predicted probabilities were found.')


    # plt.title('Confusion Matrix for ' + title + '. \n Accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '%')
    plt.title(title + '. \n Accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '%')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.yticks(rotation=0)

    set_plot_style(style_num = 9)

    if save:
        plt.savefig('confusion_matrix_' + file_name + ' with accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '.jpg', dpi=300, bbox_inches='tight')

    return cm_img, accuracy_in_test_set, y_predicted_labels

In [ ]:
def from_conf_matrix_plot_conf_matrix(conf_matrix, algorithm_type, accuracy_in_test_set, title, file_name, target_names, save=False, normalize=False, check_probabilities=False):
    cm = conf_matrix
    if normalize: 
        cmn = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        # Debugging: Print the raw confusion matrix
        print("Raw confusion matrix:\n", cm)
        cm_img = plt.figure(figsize=(4, 4))
        sn.heatmap(cmn, annot=True, fmt='.2f', xticklabels=target_names,
                   yticklabels=target_names, cmap='Blues', cbar=False)
    else:
        cm_img = plt.figure(figsize=(4, 4))
        sn.heatmap(cm, annot=True, fmt='.2f', xticklabels=target_names,
                   yticklabels=target_names, cmap='Blues', cbar=False)

    if check_probabilities == True and algorithm_type != 'SVM': #Check if the predicted probabilities sum up to 1
        normal = False
        tolerance = 1e-5
        sums = np.sum(y_predicted, axis=1)
        for i in range(len(sums)):
            if abs(sums[i] - 1) > tolerance:
                print("Warning: sum of predicted probabilities for input", i, "is not close to 1:", sums[i])
            else:
                normal = True
        if normal == True:
            print('No issues with the sum of predicted probabilities were found.')


    # plt.title('Confusion Matrix for ' + title + '. \n Accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '%')
    plt.title(title + '. \n Average Accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '%')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.yticks(rotation=0)

    set_plot_style(style_num = 9)

    if save:
        plt.savefig('confusion_matrix_' + file_name + ' with accuracy: ' + str(round(100 * accuracy_in_test_set, 2)) + '.jpg', dpi=300, bbox_inches='tight')

    return cm_img, accuracy_in_test_set#, y_predicted_labels

In [35]:
def average_confusion_matrices(list_conf_matrices):
    # Sum all matrices element-wise
    sum_matrix = np.sum(list_conf_matrices, axis=0)

    # Calculate the average by dividing the sum by the number of matrices
    average_matrix = sum_matrix / len(list_conf_matrices)

    return average_matrix

## Defining functions to ROC Graphs

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_ROC_graph(model, test_data, test_labels, title, file_name, model_type, model_names, models_dict = None, multiple_models = False, save = False):

    if model_type == 'CNN' or model_type == 'NN':
        # Get the predicted probabilities for each class on the test dataset
        predicted_probs = model.predict(test_data)
        # Extract the probabilities for class 1
        probs_class_1 = predicted_probs[:, 1]
        # Calculate the false positive rate (FPR), true positive rate (TPR), and thresholds
        fpr, tpr, thresholds = roc_curve(test_labels, probs_class_1)
    elif model_type == 'SVM':
        # Get the decision function values for the test dataset
        decision_function_values = model.decision_function(test_data)
        # Calculate the false positive rate (FPR), true positive rate (TPR), and thresholds
        fpr, tpr, thresholds = roc_curve(test_labels, decision_function_values)

    # Calculate the AUC-ROC
    auc_roc = auc(fpr, tpr)

    # Plot the ROC curve
    set_plot_style(style_num = 17)
    plt.figure()
    plt.plot(fpr, tpr, color='green', lw=2, label= model_names[0] + ' curve (area = %0.2f)' % auc_roc)
    
    if multiple_models:
        colors_list = ['red', 'black', 'c', 'm', 'y', 'k']
        # for model_i in models_dict:
        for index, model_i in enumerate(models_dict):
            plt.plot(models_dict[model_i][0], models_dict[model_i][1], color=colors_list[index], lw=2, label= model_names[index + 1] + ' curve (area = %0.2f)' % models_dict[model_i][2])
    
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label = 'Random Classifier')
    
    # plt.title('Receiver Operating Characteristic (ROC) \n' + title)
    plt.title('ROC ' + title)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")

    if save:
        plt.savefig('ROC_' + file_name + '.jpg', dpi=300)
   
    return fpr, tpr, auc_roc

### Multiclass ROC: micro-average ROC curve

In [37]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
#This function calculates the micro-average ROC Curve, which works for a multiclass problem. This function only plots the
#Multi-average ROC Curve.

def plot_multiclass_micro_ROC_graph(model, test_data, test_labels, n_classes, title, file_name, model_type, model_names, models_dict=None, multiple_models=False, save=False):

    # Binarize the output
    test_labels_bin = label_binarize(test_labels, classes=[i for i in range(n_classes)])

    if model_type == 'CNN' or model_type == 'NN':
        # Get the predicted probabilities for each class on the test dataset
        y_score = model.predict(test_data)
    elif model_type == 'SVM':
        # Get the decision function values for the test dataset
        y_score = model.decision_function(test_data)

    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(test_labels_bin.ravel(), y_score.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    set_plot_style(style_num = 17)
    plt.figure()
    plt.plot(fpr["micro"], tpr["micro"], color='green', lw=2, label= model_names[0] + ' ROC curve (area = %0.2f)' % roc_auc["micro"])

    if multiple_models:
        colors_list = ['red', 'black', 'c', 'm', 'y', 'k']
        for index, model_i in enumerate(models_dict):
            plt.plot(models_dict[model_i][0], models_dict[model_i][1], color=colors_list[index], lw=2, label= model_names[index + 1] + ' ROC curve (area = %0.2f)' % models_dict[model_i][2])

    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label = 'Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Micro-average ROC ' + title)
    plt.legend(loc="lower right")

    if save:
        plt.savefig('MicroROC_' + file_name + '.jpg', dpi=300)

    return fpr["micro"], tpr["micro"], roc_auc["micro"]

## plot Multiclass ROC Curve Internal Analysis

In [38]:
from sklearn.metrics import roc_curve, roc_auc_score, auc
from sklearn.preprocessing import label_binarize

def plot_multiclass_ROC_internal_analysis(model, test_data, test_labels, n_classes, title, file_name, model_type, model_names, models_dict = None, multiple_models = False, save = False):

    #This function not only calculates and plots the micro-average ROC but it also shows how the algorithm is doing with respect to other classes being classified.
    
    # Binarize the output
    test_labels_bin = label_binarize(test_labels, classes=[0, 1, 2])
    
    if model_type == 'CNN' or model_type == 'NN':
        predicted_probs = model.predict(test_data)
    elif model_type == 'SVM':
        decision_function_values = model.decision_function(test_data)
        predicted_probs = np.exp(decision_function_values) / np.sum(np.exp(decision_function_values), axis=1, keepdims=True)

    set_plot_style(style_num=17)

    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels_bin[:, i], predicted_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(test_labels_bin.ravel(), predicted_probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    plt.figure()
    plt.plot(fpr["micro"], tpr["micro"],
             label='micro-average ROC curve (area = {0:0.2f})'
                   ''.format(roc_auc["micro"]))

    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label='ROC curve of class {0} (area = {1:0.2f})'
                                       ''.format(i, roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Some extension of Receiver operating characteristic to multi-class')
    plt.legend(loc="lower right")

    if save:
        plt.savefig('ROC_' + file_name + '.jpg', dpi=300)

    return fpr, tpr, roc_auc

In [ ]:
def plot_ROC_from_saved_ROC_vect(title, file_name, model_names, models_dict = None, save = False):
    # This function uses a dictionary as an input to create the ROC graph of previously saved ROC values of trained 
    # algortihms. This is usefull when you do not want to calculate the FPR, TPR and AUC_ROC, because you already have them.
    # Inputs: model_names is the list of model names to be displayed in the label of the graph. models_dict is the dictionary that
    # contains the name of the trained models as keys and the arrays of the ROC values.

    set_plot_style(style_num=17)    
    colors_list = ['green', 'red', 'black', 'm', 'y', 'k', 'c']
    # for model_i in models_dict:
    for index, model_i in enumerate(models_dict):
        # plt.plot(models_dict[model_i][0], models_dict[model_i][1], color=colors_list[index], lw=2, label= model_i + ' ROC curve (area = %0.2f)' % models_dict[model_i][2])
        plt.plot(models_dict[model_i][0], models_dict[model_i][1], color=colors_list[index], lw=2, label= model_names[index] + ' (acc = %0.2f)' % models_dict[model_i][2])
    
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label = 'Random Classifier')
    
    plt.title('ROC ' + title)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")

    if save:
        plt.savefig('ROC_' + file_name + '.jpg', dpi=300)

## Import Saved Model and Predict

In [ ]:
def use_prediction_model(imported_model, model_type, dataset, time_frame, data_labels):
    
    label_mapping = {0.0: data_labels[0], 1.0: data_labels[1]}

    if model_type.upper() == 'SVM':
        predict_images, predict_vmin, predict_vmax = get_all_stft_spectrogram_rgb_arrays(signals_array = dataset, 
            tf=time_frame, nperseg_val = 2**9, noverlap_val = 128, normalize = False, 
            max_freq = 30, log_scale = False, same_color_scale=True, find_scale = 'average_minmax',
            cmap_type='jet')
        predict_images_re = predict_images.reshape(predict_images.shape[0], predict_images.shape[1]*predict_images.shape[2]*predict_images.shape[3])

        predict_model = load(imported_model)
        predictions = model_svc.predict(test_images_re)
        predict_dictionary = {label_mapping[key]: value for key, value in zip(unique, counts)}
    
    return predict_dictionary

## K-Folf validation

In [ ]:
from sklearn.model_selection import KFold

def get_splitted_arrays_for_Kfold(signals_array, num_Kfolds):
    # This function takes an array with signals and uses the Kfold method to get the indices that could devide
    # the data into training and testing sets according to needed the Kfold distribution. Important: This function must be
    # used with arrays containing the same type of bacteria, as it calculates the indexes of an array that would be part of
    # the trainind and testing set of the same kind of data.  
    # Inputs:  signals_array is an array containing signals of the same type of bacteria. The `num_Kfolds` variable represents
    #          the number of times the complete dataset is divided into training and testing sets.

    
    kf = KFold(n_splits=num_Kfolds)
    kf.get_n_splits(signals_array)

    train_indices = []
    test_indices = []

    for train_index, test_index in kf.split(signals_array): # Iterate over each split and store the indices
        train_indices.append(train_index)
        test_indices.append(test_index)
    
    splitted_train_array = signals_array[train_index]
    splitted_test_array = signals_array[test_index]

    return  splitted_train_array, splitted_test_array

In [ ]:
def split_data_into_train_test(list_of_cases, number_of_folds, suffle_datasets = False, summary = True):
    # This function takes a list containing arrays, where each array is a signals_array case For each case, the function splits the 
    # array into training and testing sets based on the specified number of folds. It can handle multiple cases simultaneously. The process 
    # involves creating individual training and test sets for each case. Once this is done, the function merges all the training sets from 
    # different cases into a single, unified array named train_data. Similarly, it combines all the test sets into one array called test_data. 
    # Along with these merged arrays, the function also gives all the their corresponding labels starting from 0 up to the needed value.

    #Inputs: list_of_cases is a list containing arrays of measurements cases like ecoli_control_array or saureus_control_array.
    #        number_of_folds is the number of times one wants to divite each invididual case for creating the training and testing sets
    #        suffle_datasets is a boolean option that suffles the order within the invididual measurement cases if needed.
    #        summary: Is a boolean option that displays a summary of how the data was splitted.    
    #Returns: train_data, test_data, train_labels, test_labels regardless of the amount of cases used in the function.   

    train_data = np.empty((0, list_of_cases[0].shape[1]))  #Initialize train_data, assuming all dataset arrays have the same shape
    test_data = np.empty((0, list_of_cases[0].shape[1]))  #Initialize test_data, assuming all dataset arrays have the same shape

    train_labels = np.array([])
    test_labels = np.array([])

    for case_num in range(len(list_of_cases)):
        if suffle_datasets:
            case_temp = suffle_index(list_of_cases[case_num]) #to have all the dataset suffled.
        else:
            case_temp = list_of_cases[case_num] #to have all the dataset without shuffling.

        train_case_temp, test_case_temp = get_splitted_arrays_for_Kfold(case_temp, num_Kfolds = number_of_folds)
        case_temp_train_index, case_temp_test_index = train_case_temp.shape[0], test_case_temp.shape[0]

        train_data = np.vstack((train_data, train_case_temp)) #to append train_case_temp to train_data using vstack
        test_data = np.vstack((test_data, test_case_temp)) #to append train_case_temp to train_data using vstack

        train_labels = np.concatenate((train_labels, np.full(train_case_temp.shape[0], case_num)))
        test_labels = np.concatenate((test_labels, np.full(test_case_temp.shape[0], case_num)))

    if summary:
        print('A dataset of ' + str(train_data.shape[0]+test_data.shape[0]) + ' measurements is used.')
        print('')
        print(str(train_data.shape[0]) + ' are used for training using a K-fold with ' + str(number_of_folds) + ' folds where:')
        
        for case_num in range(len(list_of_cases)):
            print('Case ' + str(case_num+1) + ' in traning has ' + str(np.count_nonzero(train_labels == case_num)) + ' cases.')
        
        print('')
        print(str(train_data.shape[0]) + ' are used for training using a K-fold with ' + str(number_of_folds) + ' folds where:')
        
        for case_num in range(len(list_of_cases)):
            print('Case ' + str(case_num+1) + ' in testing has ' + str(np.count_nonzero(test_labels == case_num)) + ' cases.')

    return train_data, test_data, train_labels, test_labels

## Function to perform multiple trainings with either SVM or CNN

### Create CNN architecture for train_algorithms function

In [ ]:
def create_standard_CNN_architecture(train_images, cases_to_analyse):
    model_CNN = models.Sequential()

    model_CNN.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(train_images.shape[1], train_images.shape[2], train_images.shape[3])))
    model_CNN.add(layers.MaxPooling2D((2, 2)))

    model_CNN.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model_CNN.add(layers.MaxPooling2D((2, 2)))

    model_CNN.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model_CNN.add(layers.MaxPooling2D((2, 2)))

    # model_CNN.add(layers.Conv2D(64, (3, 3), activation='relu'))
    # model_CNN.add(layers.MaxPooling2D((2, 2)))

    model_CNN.add(layers.Flatten())
    model_CNN.add(layers.Dense(64, activation='relu'))
    # model_CNN.add(layers.Dense(2, activation='sigmoid'))
    if len(cases_to_analyse) == 2:
        model_CNN.add(layers.Dense(2, activation='softmax')) #In binary should be sigmoid
    if len(cases_to_analyse) > 2:
        model_CNN.add(layers.Dense(len(cases_to_analyse), activation='softmax'))
    model_CNN.summary()
    return model_CNN

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
def create_image_aug_CNN_architecture(train_images, cases_to_analyse):
    
    model_CNN = models.Sequential()

    model_CNN.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(train_images.shape[1], train_images.shape[2], train_images.shape[3])))
    model_CNN.add(layers.MaxPooling2D((2, 2)))
    model_CNN.add(layers.Dropout(0.2))

    model_CNN.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model_CNN.add(layers.MaxPooling2D((2, 2)))
    model_CNN.add(layers.Dropout(0.2))

    model_CNN.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model_CNN.add(layers.MaxPooling2D((2, 2)))
    model_CNN.add(layers.Dropout(0.2))

    # model_CNN.add(layers.Conv2D(64, (3, 3), activation='relu'))
    # model_CNN.add(layers.MaxPooling2D((2, 2)))
    # model_CNN.add(layers.Dropout(0.2))

    model_CNN.add(layers.Flatten())
    model_CNN.add(layers.Dense(64, activation='relu'))
        
    if len(cases_to_analyse) == 2:
        model_CNN.add(layers.Dense(2, activation='softmax')) #In binary should be sigmoid
    if len(cases_to_analyse) > 2:
        model_CNN.add(layers.Dense(len(cases_to_analyse), activation='softmax'))

    model_CNN.summary()

    # Add data augmentation
    datagen = ImageDataGenerator(
        rotation_range=10,
        zoom_range=0.1,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True)

    datagen.fit(train_images)



    return model_CNN, datagen

In [ ]:
def train_algorithms(algorithm_type, 
                    number_of_folds, cases_to_analyse, SVM_type='none', poly_degree='none', C_value='none',
                    learning_rate_val = 'none', number_epochs = 'none', limit_Pxx_height = None,
                    suffle_datasets = True, summary_splitted_data = False, 
                    average_results = True, plot_conf_matrix = False, normalize = False):
    
    temp_accuracy_in_test_set = []
    list_conf_matrices = []
    list_of_models = []
    temp_history_CNN_list = []

    for training_instance in range(number_of_folds-1):
        train_data, test_data, train_labels, test_labels = split_data_into_train_test(cases_to_analyse, number_of_folds= training_instance+2,
                                                                                                suffle_datasets = suffle_datasets, summary = summary_splitted_data)
        train_images, train_vmin, train_vmax, train_freqs  = get_all_Pxx_matrices(train_data, 
                    tf=ecoli7744_CTR_df['Time [s]'][0], limit_Pxx_height= limit_Pxx_height, 
                    nperseg_val = 2**9, noverlap_val = ((2**9)/(2)), find_scale = 'average_minmax',
                    same_color_scale=True, log_scale=True)
                    # max_freq = 300, normalize = False, log_scale = True, same_color_scale=True, find_scale = 'average_minmax')

        test_images, test_vmin, test_vmax, test_freqs  = get_all_Pxx_matrices(test_data, 
                    tf=ecoli7744_CTR_df['Time [s]'][0], limit_Pxx_height= limit_Pxx_height, 
                    nperseg_val = 2**9, noverlap_val = ((2**9)/(2)), find_scale = 'average_minmax', 
                    same_color_scale=True, log_scale=True)
   
    ## Training section -----------------
        
        type_of_algorithm = 0 
        if algorithm_type == 'SVM':
            if C_value == 'none':
                print('ERROR: A SVM algorithm was choen without a C value. Please choose a C_value number')
                sys.exit()
            train_images_re = train_images.reshape(train_images.shape[0], train_images.shape[1]*train_images.shape[2])
            test_images_re = test_images.reshape(test_images.shape[0], test_images.shape[1]*test_images.shape[2])

            if SVM_type == 'linear':
                type_of_algorithm = 0
                model = svm.SVC(kernel=SVM_type, C=C_value)
                model.fit(train_images_re, train_labels)
                model.score(test_images_re, test_labels)
                list_of_models.append(model)
                if average_results:
                    y_predicted_labels = model.predict(test_images_re)
                    cm_temp = confusion_matrix(test_labels, y_predicted_labels)
                    temp_accuracy_in_test_set.append(model.score(test_images_re, test_labels))
                    list_conf_matrices.append(cm_temp)

            elif SVM_type == 'poly':
                if poly_degree == 'none':
                    print('Error: A SVM Poly was chosen without a poly_degree number. Please select a poly_degree value')
                    sys.exit()
                type_of_algorithm = 1
                model = svm.SVC(kernel='poly', degree = poly_degree, C=C_value)
                model.fit(train_images_re, train_labels)
                model.score(test_images_re, test_labels)
                list_of_models.append(model)
                if average_results:
                    y_predicted_labels = model.predict(test_images_re)
                    cm_temp = confusion_matrix(test_labels, y_predicted_labels)
                    temp_accuracy_in_test_set.append(model.score(test_images_re, test_labels))
                    list_conf_matrices.append(cm_temp)
        if algorithm_type == 'CNN':
            type_of_algorithm = 2
            if learning_rate_val == 'none':
                print('Error: A CNN was chosen without a learning rate number. Please select a learning_rate_val value')
            if number_epochs == 'none':
                print('Error: A CNN was chosen without a number of epochs value. Please select a number_epochs value')
            train_images = np.expand_dims(train_images, axis=-1) #This is to resize the train_images arrays as now the Pxx matrix is being used.
            model = create_standard_CNN_architecture(train_images, cases_to_analyse)
            opt = keras.optimizers.Adam(learning_rate=learning_rate_val)
            model.compile(optimizer=opt, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
                          metrics=['accuracy'])
            temp_history_CNN = model.fit(train_images, train_labels, epochs=number_epochs,
                    validation_data=(test_images, test_labels))
            temp_history_CNN_list.append(temp_history_CNN)
            list_of_models.append(model)
            if average_results:
                y_predicted = model.predict(test_images)
                y_predicted_labels = [np.argmax(i) for i in y_predicted]
                cm_temp = confusion_matrix(test_labels, y_predicted_labels)
                
                eval_model = model.evaluate(test_images, test_labels, verbose=0)
                temp_accuracy_in_test_set.append(eval_model[1])
                list_conf_matrices.append(cm_temp)
        if algorithm_type == 'CNN_image_aug':
            type_of_algorithm = 3
            if learning_rate_val == 'none':
                print('Error: A CNN was chosen without a learning rate number. Please select a learning_rate_val value')
            if number_epochs == 'none':
                print('Error: A CNN was chosen without a number of epochs value. Please select a number_epochs value')
            train_images = np.expand_dims(train_images, axis=-1) #This is to resize the train_images arrays as now the Pxx matrix is being used.
            model, datagen = create_image_aug_CNN_architecture(train_images, cases_to_analyse)
            
            opt = keras.optimizers.Adam(learning_rate=learning_rate_val) 

            model.compile(optimizer=opt, 
                        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), metrics=['accuracy'])

            temp_history_CNN = model.fit(datagen.flow(train_images, train_labels, batch_size=32),
                                                    epochs=number_epochs,
                                                    validation_data=(test_images, test_labels))
            temp_history_CNN_list.append(temp_history_CNN)
            list_of_models.append(model)
            if average_results:
                y_predicted = model.predict(test_images)
                y_predicted_labels = [np.argmax(i) for i in y_predicted]
                cm_temp = confusion_matrix(test_labels, y_predicted_labels)
                
                eval_model = model.evaluate(test_images, test_labels, verbose=0)
                temp_accuracy_in_test_set.append(eval_model[1])
                list_conf_matrices.append(cm_temp)
        
        #Important error checks:            
        if algorithm_type == 'SVM' and SVM_type == 'none':
            print('ERROR: You have chosen a SVM algorithm but a type of SVM algorithm was not specified.')
            sys.exit() #It stops the code in case I selected SVM but I did not select the type of SVM. 
        
        print('End of training number ' + str(training_instance+1) + ' of the algorithm')
        print('')
    # if algorithm_type == 'SVM':
    average_acc_models = (sum(temp_accuracy_in_test_set))/(len(temp_accuracy_in_test_set)) #Calculation of the averages of the overall accuracy
    accuracy_information = [average_acc_models, temp_accuracy_in_test_set]

    average_matrix = average_confusion_matrices(list_conf_matrices) #Calculation of the averages of the confusion matrices 

    if type_of_algorithm == 0:
        print('An SVM linear algorithm has been used with a regularization parameter C=' + str(C_value))
    elif type_of_algorithm == 1:
        print('An SVM Poly algorithm of degree ' + str(poly_degree) + ' has been used with a regularization parameter C=' + str(C_value))
    elif type_of_algorithm == 2:
        print('An CNN algorithm has been used with a learning rate of ' + str(learning_rate_val))
    elif type_of_algorithm == 3:
        print('An CNN algorithm with image augmentation has been used with a learning rate of ' + str(learning_rate_val))

    return list_of_models, accuracy_information, average_matrix, temp_history_CNN_list